In [1]:
import plotly.io as pio
pio.renderers.default = "iframe"

from IPython.display import display, HTML

# Remove maximum height limits and force auto-resizing
display(HTML("<style>.jp-OutputArea-child { max-height: none !important; }</style>"))
display(HTML("<style>.output_subarea { max-height: none !important; }</style>"))


In [2]:
import pandas as pd
import networkx as nx
import plotly.graph_objects as go

# 1. Load your CSV files (replace file paths with your actual filenames)
nodes_df = pd.read_csv('custom/50_5_5_10_5_10_Nodes.csv',index_col=False)
edges_df = pd.read_csv('custom/50_5_5_10_5_10_Edges.csv',index_col=False)

In [3]:
# 2. Clean data types: Convert string coordinates to numeric floats
nodes_df['X'] = pd.to_numeric(nodes_df['X'])
nodes_df['Y'] = pd.to_numeric(nodes_df['Y'])
nodes_df['Z'] = pd.to_numeric(nodes_df['Z'])

In [4]:
# Ensure IDs match perfectly by stripping potential spaces and casting to string
nodes_df['NodeId'] = nodes_df['NodeId'].astype(str).str.strip()
edges_df['nodeFrom'] = edges_df['nodeFrom'].astype(str).str.strip()
edges_df['nodeTo'] = edges_df['nodeTo'].astype(str).str.strip()

In [5]:
# 3. Create a dictionary mapping NodeId -> (X, Y, Z)
pos_dict = nodes_df.set_index('NodeId')[['X', 'Y', 'Z']].to_dict('index')

# 4. Initialize NetworkX Graph and add edges
G = nx.Graph()  # Using undirected graph since they are all bidirectional
for _, row in edges_df.iterrows():
    G.add_edge(row['nodeFrom'], row['nodeTo'])

In [6]:
# 5. Extract Edge Coordinates for Plotly
edge_x = []
edge_y = []
edge_z = []

for edge in G.edges():
    start_node, end_node = edge
    # Only draw the edge if both nodes exist in your nodes data
    if start_node in pos_dict and end_node in pos_dict:
        # Start coordinate
        edge_x.extend([pos_dict[start_node]['X'], pos_dict[end_node]['X'], None])
        edge_y.extend([pos_dict[start_node]['Y'], pos_dict[end_node]['Y'], None])
        edge_z.extend([pos_dict[start_node]['Z'], pos_dict[end_node]['Z'], None])

In [7]:
# 6. Extract Node Coordinates for Plotly
node_x = []
node_y = []
node_z = []
node_text = []

for node in G.nodes():
    if node in pos_dict:
        node_x.append(pos_dict[node]['X'])
        node_y.append(pos_dict[node]['Y'])
        node_z.append(pos_dict[node]['Z'])
        node_text.append(f"Node: {node}")

In [8]:
# 7. Create Plotly Traces
edge_trace = go.Scatter3d(
    x=edge_x, y=edge_y, z=edge_z,
    line=dict(width=4, color='#888'),
    hoverinfo='none',
    mode='lines'
)

node_trace = go.Scatter3d(
    x=node_x, y=node_y, z=node_z,
    mode='markers',
    hoverinfo='text',
    text=node_text,
    marker=dict(
        showscale=False,
        colorscale='YlGnBu',
        reversescale=True,
        color=[],
        size=2,
        line=dict(width=1, color='rgb(50,50,50)')
    )
)

In [9]:
# Color nodes dynamically by their degree (number of connections)
node_adjacencies = []
for node, adjacencies in G.adjacency():
    if node in pos_dict:
        node_adjacencies.append(len(adjacencies))
node_trace.marker.color = node_adjacencies

In [10]:
# Get Agent Data
agent_data = pd.read_csv('custom/50_5_5_10_5_10_StartGoalLocations.csv', index_col=False)

# 1. Clean data types for your agents dataframe
agent_data['startNodeId'] = agent_data['startNodeId'].astype(str).str.strip()
agent_data['goalNodeId'] = agent_data['goalNodeId'].astype(str).str.strip()

# 2. Extract 3D positions for agent starts and goals
agent_start_x, agent_start_y, agent_start_z, agent_start_text = [], [], [], []
agent_goal_x, agent_goal_y, agent_goal_z, agent_goal_text = [], [], [], []

for _, row in agent_data.iterrows():
    agent_id = row['agentId']
    start_node = row['startNodeId']
    goal_node = row['goalNodeId']
    
    # Map start position
    if start_node in pos_dict:
        agent_start_x.append(pos_dict[start_node]['X'])
        agent_start_y.append(pos_dict[start_node]['Y'])
        agent_start_z.append(pos_dict[start_node]['Z'])
        agent_start_text.append(f"Agent {agent_id} Start (Node: {start_node})")
        
    # Map goal position
    if goal_node in pos_dict:
        agent_goal_x.append(pos_dict[goal_node]['X'])
        agent_goal_y.append(pos_dict[goal_node]['Y'])
        agent_goal_z.append(pos_dict[goal_node]['Z'])
        agent_goal_text.append(f"Agent {agent_id} Goal (Node: {goal_node})")

In [11]:
# 3. Create Plotly traces for Agents (Star shapes or larger distinct dots)
agent_start_trace = go.Scatter3d(
    x=agent_start_x, y=agent_start_y, z=agent_start_z,
    mode='markers',
    name='Agent Starts',
    hoverinfo='text',
    text=agent_start_text,
    marker=dict(size=6, color='lime', symbol='diamond', line=dict(width=1, color='black'))
)

agent_goal_trace = go.Scatter3d(
    x=agent_goal_x, y=agent_goal_y, z=agent_goal_z,
    mode='markers',
    name='Agent Goals',
    hoverinfo='text',
    text=agent_goal_text,
    marker=dict(size=6, color='crimson', symbol='circle', line=dict(width=1, color='black'))
)

In [12]:
# 8. Render the Interactive 3D Visualization
fig = go.Figure(
    data=[edge_trace, node_trace, agent_start_trace, agent_goal_trace],
    layout=go.Layout(
        title='3D Network with Agent Positions',
        showlegend=True,  # Changed to True so you can toggle agent visibility
        hovermode='closest',
        margin=dict(b=20, l=5, r=5, t=40),
        scene=dict(
            xaxis=dict(showgrid=True, zeroline=False),
            yaxis=dict(showgrid=True, zeroline=False),
            zaxis=dict(showgrid=True, zeroline=False),
        )
    )
)

fig.show()

### VDA 5050 Protocol & MQTT Integration

- The sections below replace standard trajectory interpolation with a real-time event-driven approach. We establish a compartmentalized `VDAFleetManager` to communicate via MQTT, and a separate trajectory-to-order generator for seamless integration with FLOWRRA.
   

## Final Execution Step

In [14]:
import config
from vda_environment import VDAEnvironment

# 1. Setup the Environment
cfg = config.get_config()
env = VDAEnvironment(pos_dict, G, agent_data, cfg)

# 2. Benchmark/Train Phase (Calculates routes and waits for physics to complete)
# (Change "networkx" to "flowrra" when ready!)
active_robots = env.dispatch_routes(routing_algorithm="networkx")
env.wait_for_fleet(active_robots, timeout=120)

# 3. Cleanup Phase (Extracts final results)
final_telemetry = env.close()

# 4. Visualization Phase (Only when you want to see the result!)
env.render_3d_visualization(final_telemetry, filename="final_benchmark.html")

Initializing VDA Environment & MQTT Connection...
Connected to MQTT Broker at localhost:1883 (Code Success)
Published order order_s0_20ec to uagv/v2/rikeb/s0/order
Published order order_s1_6c45 to uagv/v2/rikeb/s1/order
Published order order_s2_a49e to uagv/v2/rikeb/s2/order
Published order order_s3_f5d8 to uagv/v2/rikeb/s3/order
Published order order_s4_7e3e to uagv/v2/rikeb/s4/order
Published order order_s5_3247 to uagv/v2/rikeb/s5/order
Published order order_s6_65b4 to uagv/v2/rikeb/s6/order
Published order order_s7_3f17 to uagv/v2/rikeb/s7/order
Published order order_s8_c99a to uagv/v2/rikeb/s8/order
Published order order_s9_122f to uagv/v2/rikeb/s9/order
Published order order_s10_c8c9 to uagv/v2/rikeb/s10/order
Published order order_s11_0fcd to uagv/v2/rikeb/s11/order
Published order order_s12_7908 to uagv/v2/rikeb/s12/order
Published order order_s13_73b9 to uagv/v2/rikeb/s13/order
Published order order_s14_de1f to uagv/v2/rikeb/s14/order
Published order order_s15_4852 to uagv/v2/